# Hey Podcast — Phase 0 prototype

Topics in → fresh research → two-host script → AI voices → an MP3 you can play.

Free stack: **Gemini Flash** (free API tier) writes the research + script, **Kokoro** generates the voices, **ffmpeg** stitches the episode.

**Before you start:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*. (It also works on CPU, just slower.)

Run the cells top to bottom. The only thing you must provide is a free Gemini API key from https://aistudio.google.com/apikey


### 1. Install dependencies
(ffmpeg is already on Colab; this adds Kokoro, its English G2P, and the Gemini SDK.)

In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
!pip install -q kokoro "misaki[en]" soundfile "google-genai>=1.0.0"

### 2. Add your Gemini key
Get a free key at https://aistudio.google.com/apikey — it is pasted privately and not stored in the notebook.

In [ ]:
import os, getpass
from google import genai
from google.genai import types

os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("Gemini client ready.")

### 3. Config — your knobs
This is where you tune the show: host names, voices, length, pacing. Kokoro voices: female `af_heart`, `af_bella`; male `am_michael`, `am_adam`. `lang_code='a'` is US English, `'b'` is British.

In [ ]:
CONFIG = {
    "model":        "gemini-2.5-flash",   # free-tier workhorse (do NOT use Pro on free tier)
    "minutes":      6,                    # target episode length
    "host_a_name":  "Maya",
    "host_b_name":  "Rishad",
    "host_a_voice": "af_heart",           # speaker A (female)
    "host_b_voice": "am_michael",         # speaker B (male)
    "lang_code":    "a",                  # a = US English, b = British
    "sample_rate":  24000,                # Kokoro outputs 24 kHz
    "pause_ms":     320,                  # gap between speaker turns
}

### 4. Research the topics
Gemini + Google Search pulls fresh, factual material the hosts can talk about.

In [ ]:
def fetch_brief(topics, model=None):
    model = model or CONFIG["model"]
    topic_line = ", ".join(topics)
    prompt = (
        f"Gather the most important *recent* developments on these topics: {topic_line}.\n"
        "Write a tight factual brief of 8-12 bullet points that two podcast hosts could riff on. "
        "Prioritize what is new, with concrete facts, names, and numbers. No intro, no fluff, bullets only."
    )
    resp = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0.4,
        ),
    )
    return resp.text

### 5. Write the two-host script
The research brief becomes a lively spoken conversation. **This prompt is where the quality lives** — tweak the host personalities and rules here until it sounds like a real show.

In [ ]:
import json

def write_script(brief, minutes=None, model=None):
    minutes = minutes or CONFIG["minutes"]
    model = model or CONFIG["model"]
    a, b = CONFIG["host_a_name"], CONFIG["host_b_name"]
    prompt = f"""You are scripting one episode of "Hey Podcast", a short AI-generated show (~{minutes} min).

Two hosts:
- {a} (speaker "A"): warm, curious, asks the questions a smart listener would.
- {b} (speaker "B"): sharp, explains clearly with vivid everyday analogies.

Write a natural spoken conversation based ONLY on the brief below.
Rules:
- Open with a hook in the first line, not "welcome to the podcast".
- Real back-and-forth: reactions, quick interruptions, "wait, so...". Avoid long monologues.
- Use the facts from the brief. Do NOT invent statistics or quotes.
- Spoken style: contractions, short sentences. No stage directions, no asterisks, no markdown.
- End with a one-line takeaway from {a}.

Return ONLY valid JSON shaped like:
{{"title": "<catchy episode title>", "turns": [{{"speaker": "A", "text": "..."}}, {{"speaker": "B", "text": "..."}}]}}

BRIEF:
{brief}
"""
    resp = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.9,
        ),
    )
    raw = resp.text.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw[raw.find("{"): raw.rfind("}") + 1]
    return json.loads(raw)

### 6. Load Kokoro and synthesize a turn
First run downloads the model (~once). Each speaker gets its own voice.

In [ ]:
import numpy as np
from kokoro import KPipeline

pipeline = KPipeline(lang_code=CONFIG["lang_code"])

def synth_turn(text, voice):
    chunks = []
    for result in pipeline(text, voice=voice):
        audio = getattr(result, "audio", None)
        if audio is None:
            audio = result[2]              # older Kokoro yields (graphemes, phonemes, audio)
        if hasattr(audio, "detach"):
            audio = audio.detach().cpu().numpy()
        chunks.append(np.asarray(audio, dtype=np.float32))
    return np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.float32)

### 7. Stitch the turns into one MP3

In [ ]:
import soundfile as sf, subprocess, os

def _silence(ms):
    return np.zeros(int(CONFIG["sample_rate"] * ms / 1000), dtype=np.float32)

def stitch(segments):
    pause = _silence(CONFIG["pause_ms"])
    parts = []
    for seg in segments:
        if seg.size:
            parts.append(seg)
            parts.append(pause)
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.float32)

def export(audio, basename):
    wav, mp3 = f"{basename}.wav", f"{basename}.mp3"
    sf.write(wav, audio, CONFIG["sample_rate"])
    try:
        subprocess.run(["ffmpeg", "-y", "-i", wav, "-b:a", "96k", mp3],
                       check=True, capture_output=True)
        os.remove(wav)
        return mp3
    except Exception:
        return wav

### 8. The pipeline

In [ ]:
import re
from IPython.display import Audio, display

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")[:50] or "episode"

def generate_episode(topics, minutes=None):
    print("1/4  Researching:", ", ".join(topics))
    brief = fetch_brief(topics)
    print("2/4  Writing script...")
    script = write_script(brief, minutes)
    title, turns = script.get("title", "Hey Podcast"), script["turns"]
    print(f"     '{title}' - {len(turns)} turns")
    print("3/4  Synthesizing voices...")
    voice_for = {"A": CONFIG["host_a_voice"], "B": CONFIG["host_b_voice"]}
    segments = []
    for i, t in enumerate(turns, 1):
        segments.append(synth_turn(t["text"], voice_for.get(t["speaker"], CONFIG["host_a_voice"])))
        print(f"     turn {i}/{len(turns)}")
    print("4/4  Stitching + exporting...")
    path = export(stitch(segments), slugify(title))
    print("Done:", path)
    return title, path

### 9. Generate an episode
Edit the topics, run, and play it right here.

In [ ]:
topics = ["latest AI news", "space exploration this week"]   # <-- edit these

title, path = generate_episode(topics)
print("\n" + title)
display(Audio(path, autoplay=False))

---
### What to do next

This is **Phase 0** — its only job is to answer one question: *would you actually listen to this?*

If the audio sounds good, the highest-leverage knobs are:
- **The script prompt (cell 5)** — host personalities, pacing, and the "rules" decide whether it sounds like a real show or a robot reading bullet points.
- **Voices (cell 3)** — try different Kokoro voices for A and B until the chemistry feels right.
- **Length (cell 3)** — shorter episodes are cheaper and often punchier.

Once it's genuinely enjoyable, the next phase wraps this exact pipeline behind a queue + storage so it can run on demand.
